# LookAgain-ML Public TensorFlow Flowers Demonstration

This workflow uses all ten encoders: ResNet50, VGG16, InceptionV3, MobileNetV2, COCO DeepLab, ADE20K SegFormer, DINOv2-B/14, ConvNeXt-B, ViT-B/16, and SigLIP 2-B/16.

The official archive contains 3,670 JPEG images. Excluding two cross-class duplicate rows leaves 3,668 observations in 3,666 duplicate-safe groups. The locked split contains 2,568 training, 550 validation, and 550 test observations. Preparation verifies the archive checksum, safely extracts it, decodes and hashes every image, excludes cross-class conflicts, groups within-class duplicates, and verifies the manifest and locked-split checksums.

The extraction script uses pinned encoder metadata and verifies image bytes and ordered identifiers. It saves one finite array per encoder and reuses only matching, hash-verified caches. It fits no prediction model. CPU and CUDA are supported; reduce the batch size if device memory is limited. An interruption can be resumed with the same verified extraction cache. An interrupted model-fit run restarts its deterministic fitting procedure; do not infer a completed run from partial outputs.

The fitting script constructs three-fold group-safe out-of-fold predictions inside training. It tunes base heads using training data, fits the all-ten stack on out-of-fold predictions and training outcomes, and compares frozen candidates on the separate validation partition. It locks decisions, refits on training plus validation, then evaluates test predictions. Twelve group-safe splits fixed in the implemented workflow assess partition sensitivity. Paired uncertainty uses 2,000 group-bootstrap draws from fixed predictions, without model refitting.

The current reference results are SigLIP 2 accuracy 0.9836 and log loss 0.0597; the all-ten out-of-fold stack has accuracy 0.9836 and log loss 0.0777. The validation-selected primary procedure is the single encoder. Equal accuracy does not imply equal probabilistic performance. Test differences are descriptive comparisons of locked fits, not an additional selection step.

Both notebooks include installation, checksums, preparation, extraction, cache reuse, fitting, aggregate validation, artifact generation, and interpretation. Full model execution is controlled by `RUN_FULL`; it is disabled initially. No Colab GPU run is claimed. Hardware-specific bitwise reproduction of representations must be checked against the recorded checkpoint and environment identities.


## Environment and repository
Open from an extracted repository. For Colab, upload the anonymous repository ZIP and extract it in the session working directory. Install the source snapshot; do not substitute a package with the same version string.

Optional Colab upload: use the file panel to upload and extract the repository ZIP, then change the notebook working directory to the extracted repository. No identifying repository URL is embedded.

In [ ]:
from pathlib import Path
if not (Path.cwd()/"src/lookagain_ml").is_dir():
    from google.colab import files
    import zipfile, os
    uploaded=files.upload()
    archive=next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target=(Path.cwd()/member.filename).resolve()
            if not target.is_relative_to(Path.cwd().resolve()): raise RuntimeError("Unsafe archive member")
        z.extractall(Path.cwd())
    candidate=next(p for p in Path.cwd().iterdir() if p.is_dir() and (p/"src/lookagain_ml").is_dir())
    os.chdir(candidate)

In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src/lookagain_ml").is_dir()), None)
if REPO is None: raise RuntimeError("Extract the repository and open the notebook from within it")
RUN_FULL = False
WORK = REPO / "work"
DEMO = REPO / "workflow_demo/tensorflow_flowers"
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True, cwd=REPO)
print("Model execution:", "enabled" if RUN_FULL else "NOT RUN")

In [ ]:
if RUN_FULL:
    run("-m", "pip", "install", ".")
    run("-m", "pip", "install", ".[modern]")
    from lookagain_ml import nested_validation_selection
    assert callable(nested_validation_selection)
run("replication/verify_checksums.py")
identity=json.loads((REPO/"environment/source_identity.json").read_text())
assert all(hashlib.sha256((REPO/p).read_bytes()).hexdigest()==h for p,h in identity.items())
print("PASS: source identity")

## Public data, archive integrity, duplicates and locked split
The preparation script checks the archive size and SHA-256 before extraction. It verifies decoding, hashes, class conflicts, group integrity, row count, and locked split. Keep image data outside the distributable source tree.

In [ ]:
if RUN_FULL:
    run(DEMO/"analysis/prepare_flowers.py", "--working-root", WORK/"flowers", "--durable-root", WORK/"durable")
else: print("NOT RUN: public download and preparation")

## Frozen representation extraction and cache reuse
All ten encoders are extracted once. The cache identifies checkpoint, ordered observation IDs, ordered image hashes, dimensions and array bytes. Rerunning this cell reuses verified caches. Set DEVICE to `cpu` or `cuda` as appropriate; `auto` selects the available backend.

In [ ]:
DEVICE = "auto"
if RUN_FULL:
    run(DEMO/"analysis/extract_features.py", "--working-root", WORK/"flowers", "--cache-root", WORK/"features", "--model-cache-root", WORK/"models", "--device", DEVICE, "--batch-size", "8")
else: print("NOT RUN: encoder extraction")

## Training-only tuning, validation-based selection and locked refit
The runner fits the all-ten out-of-fold stack within training, compares it with the selected single encoder on validation, freezes decisions, refits on training plus validation, and only then evaluates the test set. The same procedure is used for twelve repeated group-safe splits.

In [ ]:
if RUN_FULL:
    run(DEMO/"analysis/run_flowers.py", "--package-root", REPO, "--split-csv", WORK/"flowers/splits/tf_flowers_locked_split_v1.csv", "--cache-root", WORK/"features", "--output", WORK/"results", "--repetitions", "12")
else: print("NOT RUN: fixed and repeated model fitting")

## Paired uncertainty, tables and figures
The published aggregate comparison uses 2,000 paired group-bootstrap draws from fixed predictions. Plotting released aggregates verifies their presentation, independently of a full model run.

In [ ]:
if RUN_FULL:
    run(DEMO/"analysis/summarize_run.py", "--results", WORK/"results")
else: print("NOT RUN: paired uncertainty from a fresh model run")

In [ ]:
run(DEMO/"analysis/verify_reference_results.py")
run(DEMO/"analysis/generate_flowers_artifacts.py")

## Interpretation and execution record
Reference SigLIP 2 test accuracy: **0.9836**; log loss: **0.0597**. All-ten out-of-fold stack accuracy: **0.9836**; log loss: **0.0777**. Twelve-split variability and fixed-test conditional intervals describe different uncertainty sources. The notebook contains no saved model-execution outputs. Validate fresh run results separately before claiming reproduction.

## Validation and data access

The delivered Colab and Jupyter notebooks have static JSON and Python-syntax validation. Earlier successful executions used related workflows. The exact delivered notebooks were not executed end to end during this packaging stage; full GPU execution is not claimed.

Saved-result verification covers retained predictions and selected numerical metrics, displayed result rows, cost rows and scalar definitions. Package tests, compilation and privacy checks are separate checks, not one universal artifact-reload certificate.

Cache hashes and verification records are retained alongside saved metrics and predictions. Cache availability must be checked before attempting a reproduction; a regenerated array must not be represented as the original archived array.

Both repository releases exclude restricted images, row-level metadata, private filesystem paths, predictions, embeddings, fitted models and representation arrays. Obtain restricted datasets independently under their applicable access terms. A private repository does not itself authorize disclosure, and removing photographs alone does not establish anonymity of derivatives. The code license does not grant rights to datasets. The TensorFlow Flowers workflow remains separate and retains its attribution and license information.
